<a href="https://colab.research.google.com/github/Hiroj12b/Cn6005/blob/main/w7Naive%20Bayes%20Tutorial%20(Colab).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# ============================================
# WEEK 7 LAB - Naive Bayes Tutorial (Colab)
# ============================================
# This notebook shows how to do your Week 7 Naive Bayes tasks in Python.
#
# We use a simple helper that:
# - Takes prior probabilities P(C)
# - Takes conditional probabilities P(feature=value | C)
# - Computes P(C | X) up to a proportionality constant
# - Chooses the most likely class

from functools import reduce
import operator

def naive_bayes_scores(priors, cond_probs, x_features):
    """
    Compute unnormalised Naive Bayes scores for each class.

    priors: dict, e.g. {'yes': 0.5, 'no': 0.5}
    cond_probs: dict of dicts:
        {
          'yes': {
             'feature1': {'valueA': p, 'valueB': p, ...},
             'feature2': {...}
          },
          'no': {...}
        }
    x_features: dict of the instance, e.g.
        {'Weather': 'Rain', 'Road': 'Good', ...}

    Returns:
        scores: dict {class: score} where
          score = P(C) * ∏ P(feature=value | C)
        posteriors: dict {class: normalised probability}
    """
    scores = {}

    for c in priors:
        # Start with prior P(C)
        score = priors[c]
        # Multiply by each likelihood P(x_i | C)
        for feat, val in x_features.items():
            # If we didn't define a probability, assume 1 (neutral)
            p = cond_probs.get(c, {}).get(feat, {}).get(val, 1.0)
            score *= p
        scores[c] = score

    # Normalise to sum to 1 (if total > 0)
    total = sum(scores.values())
    if total > 0:
        posteriors = {c: s / total for c, s in scores.items()}
    else:
        # all zero: cannot normalise
        posteriors = {c: 0.0 for c in scores}

    return scores, posteriors

def print_nb_result(title, priors, cond_probs, x_features):
    print("=" * 60)
    print(title)
    print("Instance X:", x_features)
    print("Priors:", priors)

    scores, posteriors = naive_bayes_scores(priors, cond_probs, x_features)

    print("\nUnnormalised scores (P(C) * ∏ P(x_i | C)):")
    for c, s in scores.items():
        print(f"  {c:>10}: {s:.6f}")

    print("\nNormalised posterior probabilities P(C | X):")
    for c, p in posteriors.items():
        print(f"  {c:>10}: {p:.6f}")

    best_class = max(scores, key=scores.get)
    print(f"\nPredicted class (argmax): {best_class}")
    print("=" * 60)
    print()


# ==========================================================
# TASK 1 – Accident Prediction
# X = (Rain, Good, Normal, No)
# Classes: 'yes' (accident), 'no' (no accident)
# Using the probabilities from your Week 7 document.
# ==========================================================

priors_task1 = {
    'yes': 0.5,  # P(accident = yes)
    'no':  0.5   # P(accident = no)
}

cond_task1 = {
    'yes': {
        'Weather': {'Rain': 0.2},     # P(Rain | accident=yes) = 1/5 = 0.2
        'Road':    {'Good': 0.2},     # P(Good | yes) = 1/5
        'Traffic': {'Normal': 0.2},   # P(Normal | yes) = 1/5
        'Alcohol': {'No': 0.4},       # P(No | yes) = 2/5 = 0.4
    },
    'no': {
        'Weather': {'Rain': 0.4},     # P(Rain | accident=no) = 2/5 = 0.4
        'Road':    {'Good': 0.6},     # P(Good | no) = 3/5 = 0.6
        'Traffic': {'Normal': 0.4},   # P(Normal | no) = 2/5 = 0.4
        'Alcohol': {'No': 0.8},       # P(No | no) = 4/5 = 0.8
    }
}

X_task1 = {
    'Weather': 'Rain',
    'Road': 'Good',
    'Traffic': 'Normal',
    'Alcohol': 'No'
}

print_nb_result("TASK 1 – Accident Prediction", priors_task1, cond_task1, X_task1)


# ==========================================================
# TASK 2 – Weather-Based Game Prediction (Play Tennis)
# Question 1: X = (sunny, hot, high, false)
# Classes: 'yes' (play), 'no' (don’t play)
# Using the probabilities from your Week 7 document.
# ==========================================================

priors_task2 = {
    'yes': 9/14,   # P(play = yes)
    'no':  5/14    # P(play = no)
}

cond_task2 = {
    'yes': {
        'Outlook':    {'sunny': 2/9},   # P(sunny | yes) = 2/9
        'Temperature':{'hot': 2/9},     # P(hot | yes) = 2/9
        'Humidity':   {'high': 3/9},    # P(high | yes) = 3/9
        'Windy':      {'false': 6/9},   # P(false | yes) = 6/9
    },
    'no': {
        'Outlook':    {'sunny': 3/5},   # P(sunny | no) = 3/5
        'Temperature':{'hot': 2/5},     # P(hot | no) = 2/5
        'Humidity':   {'high': 4/5},    # P(high | no) = 4/5
        'Windy':      {'false': 2/5},   # P(false | no) = 2/5
    }
}

X_task2_q1 = {
    'Outlook': 'sunny',
    'Temperature': 'hot',
    'Humidity': 'high',
    'Windy': 'false'
}

print_nb_result("TASK 2 – Play Tennis (Question 1)", priors_task2, cond_task2, X_task2_q1)


# Question 3: X' = (overcast, cool, high, true)

cond_task2_q3 = {
    'yes': {
        'Outlook':    {'overcast': 4/9},  # P(overcast | yes) = 4/9
        'Temperature':{'cool': 3/9},      # P(cool | yes) = 3/9
        'Humidity':   {'high': 3/9},      # P(high | yes) = 3/9
        'Windy':      {'true': 3/9},      # P(true | yes) = 3/9
    },
    'no': {
        'Outlook':    {'overcast': 0/5},  # P(overcast | no) = 0/5 = 0
        'Temperature':{'cool': 1/5},      # P(cool | no) = 1/5
        'Humidity':   {'high': 4/5},      # P(high | no) = 4/5
        'Windy':      {'true': 3/5},      # P(true | no) = 3/5
    }
}

X_task2_q3 = {
    'Outlook': 'overcast',
    'Temperature': 'cool',
    'Humidity': 'high',
    'Windy': 'true'
}

print_nb_result("TASK 2 – Play Tennis (Question 3)", priors_task2, cond_task2_q3, X_task2_q3)


# ==========================================================
# TASK 3 – Loan Approval Prediction
# Features: EmploymentStatus, CreditHistory, IncomeLevel
# Classes: 'yes' (LoanApproved = Yes), 'no' (LoanApproved = No)
# Using the probabilities from your Week 7 document.
# ==========================================================

priors_task3 = {
    'yes': 3/5,   # P(LoanApproved = Yes)
    'no':  2/5    # P(LoanApproved = No)
}

# First applicant: Employed, Good, Medium
cond_task3 = {
    'yes': {
        'EmploymentStatus': {'Employed': 2/3},   # P(Employed | yes) = 2/3
        'CreditHistory':    {'Good': 3/3},       # P(Good | yes) = 3/3
        'IncomeLevel':      {'Medium': 1/3},     # P(Medium | yes) = 1/3
    },
    'no': {
        'EmploymentStatus': {'Employed': 1/2},   # P(Employed | no) = 1/2
        'CreditHistory':    {'Good': 0/2},       # P(Good | no) = 0
        'IncomeLevel':      {'Medium': 1/2},     # P(Medium | no) = 1/2
    }
}

X_task3_app1 = {
    'EmploymentStatus': 'Employed',
    'CreditHistory': 'Good',
    'IncomeLevel': 'Medium'
}

print_nb_result("TASK 3 – Loan Approval (Applicant 1)", priors_task3, cond_task3, X_task3_app1)


# Second applicant: Unemployed, Bad, Low
cond_task3_app2 = {
    'yes': {
        'EmploymentStatus': {'Unemployed': 1/2},  # P(Unemployed | yes) = 1/2
        'CreditHistory':    {'Bad': 0/2},         # P(Bad | yes) = 0
        'IncomeLevel':      {'Low': 1/2},         # P(Low | yes) = 1/2
    },
    'no': {
        'EmploymentStatus': {'Unemployed': 1/2},  # P(Unemployed | no) = 1/2
        'CreditHistory':    {'Bad': 2/2},         # P(Bad | no) = 2/2
        'IncomeLevel':      {'Low': 1/2},         # P(Low | no) = 1/2
    }
}

X_task3_app2 = {
    'EmploymentStatus': 'Unemployed',
    'CreditHistory': 'Bad',
    'IncomeLevel': 'Low'
}

print_nb_result("TASK 3 – Loan Approval (Applicant 2)", priors_task3, cond_task3_app2, X_task3_app2)


# ==========================================================
# TASK 4 – Disease Diagnosis
# Features: fever, cough, fatigue, travelhistory
# Classes: 'positive', 'negative'
# Using the probabilities from your Week 7 document.
# ==========================================================

priors_task4 = {
    'positive': 3/5,   # P(positive) = 0.6
    'negative': 2/5    # P(negative) = 0.4
}

# Question 1: X = (fever=yes, cough=no, fatigue=yes, travelhistory=no)
cond_task4_q1 = {
    'positive': {
        'fever':         {'yes': 3/3},  # P(fever=yes | positive) = 3/3 = 1
        'cough':         {'no': 1 - 2/3}, # from doc: P(cough=yes | positive)=2/3 → cough=no = 1/3
        'fatigue':       {'yes': 3/3},  # = 1
        'travelhistory': {'no': 1 - 2/3}, # P(travelhistory=yes | positive)=2/3 → no = 1/3
    },
    'negative': {
        # for this first question we only need positive, but we'll still define something
        'fever':         {'yes': 0},       # not used in doc for Q1, but put placeholder
        'cough':         {'no': 0},
        'fatigue':       {'yes': 0},
        'travelhistory': {'no': 0},
    }
}

X_task4_q1 = {
    'fever': 'yes',
    'cough': 'no',
    'fatigue': 'yes',
    'travelhistory': 'no'
}

print_nb_result("TASK 4 – Disease Diagnosis (Question 1, using doc positive likelihoods)",
                priors_task4, cond_task4_q1, X_task4_q1)


# For a fuller version, define both classes exactly as in your text:

cond_task4_full = {
    'positive': {
        'fever':         {'yes': 3/3, 'no': 0},   # 3/3 = 1
        'cough':         {'yes': 2/3, 'no': 1/3},
        'fatigue':       {'yes': 3/3, 'no': 0},
        'travelhistory': {'yes': 2/3, 'no': 1/3},
    },
    'negative': {
        'fever':         {'yes': 2/2, 'no': 0},   # 2/2 = 1
        'cough':         {'yes': 2/2, 'no': 0},
        'fatigue':       {'yes': 2/2, 'no': 0},
        'travelhistory': {'yes': 1/2, 'no': 1/2},
    }
}

# Question 1 again with full model
print_nb_result("TASK 4 – Disease Diagnosis (Question 1, full model)",
                priors_task4, cond_task4_full, X_task4_q1)

# Question 2: X = (fever=no, cough=yes, fatigue=no, travelhistory=no)
X_task4_q2 = {
    'fever': 'no',
    'cough': 'yes',
    'fatigue': 'no',
    'travelhistory': 'no'
}

print_nb_result("TASK 4 – Disease Diagnosis (Question 2, full model)",
                priors_task4, cond_task4_full, X_task4_q2)




TASK 1 – Accident Prediction
Instance X: {'Weather': 'Rain', 'Road': 'Good', 'Traffic': 'Normal', 'Alcohol': 'No'}
Priors: {'yes': 0.5, 'no': 0.5}

Unnormalised scores (P(C) * ∏ P(x_i | C)):
         yes: 0.001600
          no: 0.038400

Normalised posterior probabilities P(C | X):
         yes: 0.040000
          no: 0.960000

Predicted class (argmax): no

TASK 2 – Play Tennis (Question 1)
Instance X: {'Outlook': 'sunny', 'Temperature': 'hot', 'Humidity': 'high', 'Windy': 'false'}
Priors: {'yes': 0.6428571428571429, 'no': 0.35714285714285715}

Unnormalised scores (P(C) * ∏ P(x_i | C)):
         yes: 0.007055
          no: 0.027429

Normalised posterior probabilities P(C | X):
         yes: 0.204583
          no: 0.795417

Predicted class (argmax): no

TASK 2 – Play Tennis (Question 3)
Instance X: {'Outlook': 'overcast', 'Temperature': 'cool', 'Humidity': 'high', 'Windy': 'true'}
Priors: {'yes': 0.6428571428571429, 'no': 0.35714285714285715}

Unnormalised scores (P(C) * ∏ P(x_i | C)):
